# CymbalGoal AlloyDB Workshop—Index Benchmark: ScaNN vs HNSW under load

**Track A, pipeline step 03.** Answers one question for the Module 2 demo: at 38,445 scouting
reports a single search takes about 13 ms with either index, so does the difference between
**ScaNN** and **HNSW** show up when **dozens to hundreds of searches run at once**?

| Step | What happens |
| :-- | :-- |
| **Corpus** | loads the 38,445 reports into their own database (parallel `COPY`, ~30 s), unless already there |
| **Queries** | embeds 96 scout-style searches once with `ai.embedding()`, and computes each one's exact top ten (with and without a filter) |
| **Indexes** | builds ScaNN and HNSW on separate copies of the table; records build time and size. HNSW indexes 3,072 dimensions only in half precision (`halfvec`) |
| **Match recall** | tunes each index (`scann.num_leaves_to_search`, `hnsw.ef_search`) to the smallest setting that reaches the target recall, so neither wins by doing less work |
| **Filtered recall** | the same searches restricted to one season and `SIGN`, the case where the two indexes differ most |
| **Load** | 1 → 16 → 64 → 128 concurrent searches per index, plain and filtered: queries per second, p50 / p95 / p99 latency |

The searches pull their query vector from a table inside the database, so the benchmark measures the
index, not the embedding model or the network. Run all; the last cell is the handback.

---
## 1 · Configuration

In [ ]:
RUN_INDEXES  = True     # build both indexes (skipped if they already exist; set REBUILD = True to time again)
REBUILD      = False
RUN_TUNING   = True     # match both indexes to TARGET_RECALL
RUN_LOAD     = True     # the concurrency benchmark
CLIENT_STEPS = [1, 16, 64, 128]
STEP_SECONDS = 40       # measured window per step, after a 5-second warm-up
TARGET_RECALL = 0.95

In [ ]:
import os, sys, subprocess

if not os.environ.get("CG_FIXTURE"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "google-cloud-alloydb-connector[pg8000]", "pandas"], check=True)
print("client libraries ready")

In [ ]:
# ---- Where things live ---------------------------------------------------------------------
GCS_READ       = "gs://class-demo/alloydb-labs/cymbalgoal"                       # read-only
GCS_WORK       = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop"
STAGED_PREFIX  = f"{GCS_WORK}/scout-reports/staged"
RESULTS_PREFIX = f"{GCS_WORK}/benchmarks"                                       # the only course-bucket write
DB_NAME        = "cymbalgoal_bench"
EMBED_MODEL    = "gemini-embedding-001"
MAINTENANCE_WORK_MEM = "2GB"

# ---- Index settings ------------------------------------------------------------------------
HNSW_M, HNSW_EF_CONSTRUCTION = 16, 64            # pgvector defaults
SCANN_SWEEP = [10, 20, 39, 59, 78, 118]           # leaves searched (of ~196): 5% … 60%
HNSW_SWEEP  = [40, 64, 100, 160, 250, 400]        # hnsw.ef_search

# ---- 96 scout-style searches: 12 roles × 8 traits ------------------------------------------
ROLES = ["goalkeeper", "centre-back", "left-back", "right-back", "defensive midfielder",
         "central midfielder", "attacking midfielder", "left winger", "right winger",
         "centre-forward", "second striker", "wing-back"]
TRAITS = ["young and improving", "experienced and reliable", "dominant in the air",
          "quick on the counter-attack", "creative in tight spaces", "a regular goal threat",
          "disciplined without the ball", "inconsistent but talented"]
QUERY_TEXTS = [f"a {r} who is {t}" for r in ROLES for t in TRAITS]

---
## 2 · Environment and guard

In [ ]:
import csv, gzip, io, json, math, re, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

_FIXTURE = os.environ.get("CG_FIXTURE")          # offline tests only; unset in Colab


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


BASE = Path("/content/cymbalgoal-loadtest")
try:
    BASE.mkdir(parents=True, exist_ok=True)
except OSError:
    BASE = Path.home() / "cymbalgoal-loadtest"
    BASE.mkdir(parents=True, exist_ok=True)

if _FIXTURE:
    PROJECT, USER, REGION = "fixture-project", "postgres", "us-central1"
    CLUSTER, INSTANCE, PROJECT_NUMBER = "cymbalgoal-cluster", "cymbalgoal-primary", "123"
    SHAPE = {"cpu": 8, "availability": "ZONAL", "version": "POSTGRES_18"}
else:
    PROJECT = sh("gcloud config get-value project")
    USER = sh("gcloud config get-value account")
    PROJECT_NUMBER = sh(f"gcloud projects describe {PROJECT} --format='value(projectNumber)'")
    clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
    assert clusters, "No AlloyDB cluster in this project. Apply the Terraform first."
    _cl = clusters[0]
    CLUSTER = _cl["name"].split("/")[-1]
    REGION = _cl["name"].split("/locations/")[1].split("/")[0]
    _inst = [i for i in json.loads(sh(f"gcloud alloydb instances list --cluster={CLUSTER} "
                                      f"--region={REGION} --format=json") or "[]")
             if i.get("instanceType") == "PRIMARY"]
    assert _inst, f"No PRIMARY instance in cluster {CLUSTER}."
    INSTANCE = _inst[0]["name"].split("/")[-1]
    SHAPE = {"cpu": _inst[0].get("machineConfig", {}).get("cpuCount"),
             "availability": _inst[0].get("availabilityType"),
             "version": _cl.get("databaseVersion")}

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
PROJECT_BUCKET = f"gs://{PROJECT}-cymbalgoal-staging"
SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-alloydb.iam.gserviceaccount.com"

# ---- Guarded Cloud Storage -----------------------------------------------------------------
GCS_LOG = []


def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")


READ_OK = (GCS_READ, GCS_WORK, PROJECT_BUCKET)
WRITE_OK = (RESULTS_PREFIX, PROJECT_BUCKET)


def _guard(args):
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if verb in ("ls", "cat"):
        bad = [u for u in uris if not any(_under(u, p) for p in READ_OK)]
    elif verb == "cp":
        if not any(_under(args[-1], p) for p in WRITE_OK):
            raise PermissionError(f"gcs(): writes only under {WRITE_OK}, refused {args[-1]}")
        bad = [u for u in uris[:-1] if not any(_under(u, p) for p in READ_OK)]
    else:
        raise PermissionError(f"gcs(): '{verb}' refused—only ls, cat and cp are allowed")
    if bad:
        raise PermissionError(f"gcs(): outside the allowed prefixes: {bad}")
    GCS_LOG.append("gcloud storage " + " ".join(args))


def _fx(uri):
    return Path(_FIXTURE) / uri[len("gs://"):]


def gcs(*args):
    _guard(args)
    if _FIXTURE:
        import shutil
        if args[0] == "cat":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1,
                                               p.read_text() if p.exists() else "", "" if p.exists() else "No URLs matched")
        if args[0] == "ls":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1, args[-1] if p.exists() else "", "")
        d = _fx(args[-1]); d.parent.mkdir(parents=True, exist_ok=True)
        src = args[-2]
        shutil.copy(_fx(src) if src.startswith("gs://") else src, d)
        return subprocess.CompletedProcess(args, 0, "", "")
    return subprocess.run(["gcloud", "storage", *args], capture_output=True, text=True)


def gcs_stream(uri):
    _guard(("cat", uri))
    if _FIXTURE:
        fh = open(_fx(uri), "rb")
        return fh, fh.close
    p = subprocess.Popen(["gcloud", "storage", "cat", uri], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)

    def _close():
        p.stdout.close(); p.kill(); p.wait()
    return p.stdout, _close


def get_text(uri):
    r = gcs("cat", uri)
    return r.stdout if r.returncode == 0 else None


# ---- Results and handback ------------------------------------------------------------------
HB, RESULTS = {}, {}


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def mins(s):
    if s is None:
        return "—"
    return f"{s:.0f} s" if s < 90 else f"{s / 60:.1f} min"


print(f"project   {PROJECT}  (#{PROJECT_NUMBER})")
print(f"cluster   {CLUSTER}  {REGION}  {SHAPE}")
print(f"instance  {INSTANCE}")
print(f"you       {USER}")
print(f"runtime   {os.cpu_count()} vCPU")
hb("2 · Environment", f"project {PROJECT}; cluster {CLUSTER} {REGION}; instance {INSTANCE} {SHAPE}; "
                      f"you {USER}; runtime {os.cpu_count()} vCPU")

---
## 3 · Connect and create the benchmark database

In [ ]:
if _FIXTURE:
    import pg8000.dbapi

    def _raw_connect(db):
        return pg8000.dbapi.connect(user="postgres", host=os.environ.get("CG_PGHOST", "localhost"),
                                    port=int(os.environ.get("CG_PGPORT", "5432")), database=db)
else:
    from google.cloud.alloydb.connector import Connector, IPTypes
    connector = Connector()

    def _raw_connect(db):
        return connector.connect(INSTANCE_URI, "pg8000", user=USER, db=db,
                                 enable_iam_auth=True, ip_type=IPTypes.PUBLIC)


def connect(db=DB_NAME):
    c = _raw_connect(db)
    c.autocommit = True
    return c


_tls = threading.local()


def session():
    c = getattr(_tls, "conn", None)
    if c is not None:
        try:
            cur = c.cursor(); cur.execute("SELECT 1"); cur.close()
            return c
        except Exception:
            pass
    _tls.conn = connect()
    return _tls.conn


def _drain(conn, show=False):
    out = []
    for n in list(getattr(conn, "notices", None) or []):
        m = n.get(b"M") if isinstance(n, dict) else None
        if m:
            out.append(m.decode())
    try:
        conn.notices.clear()
    except Exception:
        pass
    if show:
        for m in out:
            print("   ", m)
    return out


def run(sql, params=None, show_notices=False):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cur.close()
    return _drain(conn, show_notices)


def q(sql, params=None):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall() if cur.description else []
    cur.close()
    _drain(conn)
    return pd.DataFrame(rows, columns=cols)


t0 = time.time()
_c = connect("postgres")
_cur = _c.cursor()
_cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
if _cur.fetchone() is None:
    _cur.execute(f'CREATE DATABASE "{DB_NAME}"')
    print(f"created database {DB_NAME}")
else:
    print(f"database {DB_NAME} already exists")
_cur.execute("SELECT version()")
print(_cur.fetchone()[0].split(" on ")[0], f"— connected in {time.time() - t0:.1f}s")
_cur.close(); _c.close()

EXTS = ["vector"] if _FIXTURE else ["vector", "alloydb_scann", "google_ml_integration"]
for ext in EXTS:
    run(f"CREATE EXTENSION IF NOT EXISTS {ext}")
if _FIXTURE:   # offline stand-in for the in-database model call: a fixed pseudo-embedding per text
    run("CREATE SCHEMA IF NOT EXISTS ai")
    run('''CREATE OR REPLACE FUNCTION ai.embedding(model text, content text) RETURNS vector
           LANGUAGE sql IMMUTABLE AS $$
             SELECT array_agg(sin(i * (abs(hashtext(content)) % 997 + 1)))::vector
             FROM generate_series(1, 3072) AS i $$''')
display(q("SELECT extname, extversion FROM pg_extension ORDER BY 1"))

---
## 4 · The corpus

Loads the staged reports into `reports` (no foreign keys: this database has no `players` or
`clubs`), using the per-season files in parallel. Skipped when `reports` already holds every row.

In [ ]:
MANIFEST = json.loads(get_text(f"{STAGED_PREFIX}/scout_reports_manifest.json") or "null")
DDL = get_text(f"{STAGED_PREFIX}/scout_reports.sql")
assert MANIFEST and DDL, f"staged corpus not found under {STAGED_PREFIX}/"
COLS = MANIFEST["column_order"]
FILES = MANIFEST["files"]
FULL = "scout_reports.csv.gz"
SEASON_FILES = sorted(k for k in FILES if re.fullmatch(r"scout_reports_\d{4}\.csv\.gz", k))
N_ROWS = FILES[FULL]["rows"]

fh, close = gcs_stream(f"{STAGED_PREFIX}/{FULL}")
with gzip.GzipFile(fileobj=fh) as gz:
    first = next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8")))
close()
assert len(first) == len(COLS), f"first row has {len(first)} fields, manifest lists {len(COLS)}"
DIM = len(first[-1].strip("[]").split(","))

print(f"prompt version {MANIFEST['prompt_version']}, staged {MANIFEST['generated_utc']}")
print(f"full file      {N_ROWS:,} rows, {FILES[FULL]['bytes'] / 1e6:,.1f} MB compressed, {DIM}-dim embeddings")
print(f"season files   {len(SEASON_FILES)}: {SEASON_FILES[0]} … {SEASON_FILES[-1]}")
print(f"columns        {', '.join(COLS)}")
hb("4 · Corpus", f"{MANIFEST['prompt_version']}; {N_ROWS:,} rows; {FILES[FULL]['bytes'] / 1e6:,.1f} MB gz full file; "
                 f"{len(SEASON_FILES)} season files; {DIM} dims")

_REF = re.compile(r"\s+REFERENCES\s+\w+\s*\(\s*\w+\s*\)", re.I)


def make_table(name, with_fk):
    # A fresh, empty copy of scout_reports under another name. Timing a load into a table that
    # already holds rows (or indexes) measures something else.
    ddl = re.sub(r"\bscout_reports\b", name, DDL)
    ddl = "\n".join(l for l in ddl.splitlines() if not l.lstrip().startswith("--"))
    if not with_fk:
        ddl = _REF.sub("", ddl)
    run(f"DROP TABLE IF EXISTS {name} CASCADE")
    run(ddl)


def copy_file(table, uri, cols=COLS, conn=None):
    # Client-side COPY: the file streams from Cloud Storage, is decompressed here, and pg8000
    # pushes it into COPY FROM STDIN. Nothing lands on this runtime's disk.
    own = conn is None
    conn = conn or connect()
    fh, close = gcs_stream(uri)
    try:
        with gzip.GzipFile(fileobj=fh) as gz:
            cur = conn.cursor()
            cur.execute(f"COPY {table} ({', '.join(cols)}) FROM STDIN WITH (FORMAT csv)", stream=gz)
            cur.close()
    finally:
        close()
        if own:
            conn.close()


def count(table):
    return int(q(f"SELECT count(*) AS n FROM {table}")["n"][0])


def verify(table):
    n = count(table)
    d = int(q(f"SELECT vector_dims(report_embedding) AS d FROM {table} LIMIT 1")["d"][0]) if n else 0
    return n, d

In [ ]:
S = "4 · Corpus"
PAR = 8
if q("SELECT to_regclass('reports') IS NOT NULL AS x")["x"][0] and count("reports") == N_ROWS:
    print(f"reports already loaded ({N_ROWS:,} rows)")
    hb(S, "already loaded")
else:
    make_table("reports", False)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=PAR) as pool:
        list(pool.map(lambda f: copy_file("reports", f"{STAGED_PREFIX}/{f}"), SEASON_FILES))
    n, d = verify("reports")
    assert n == N_ROWS, f"loaded {n:,} rows, expected {N_ROWS:,}"
    print(f"loaded {n:,} rows ({d} dims) in {time.time() - t0:.0f}s")
    hb(S, f"loaded {n:,} rows in {time.time() - t0:.0f}s")
run("ANALYZE reports")
VER = q("SELECT extname, extversion FROM pg_extension WHERE extname IN ('vector', 'alloydb_scann')")
VER = dict(zip(VER.extname, VER.extversion))
print(f"extensions: {VER}")
hb(S, f"extensions: {VER}")

---
## 5 · Query vectors and exact answers

Each of the 96 searches is embedded once and stored in `qvec`, in full and half precision. Each also
gets a **filter**: one season and `recommendation = 'SIGN'` (about 1.5% of the corpus), the kind of
search a scout actually runs. The exact top ten for both versions comes from a full scan of every row.

In [ ]:
S = "5 · Queries"
HALF = "" if _FIXTURE else "::halfvec(3072)"      # the offline test server has no halfvec
seasons = sorted(int(x) for x in q("SELECT DISTINCT season FROM reports").season)

if q("SELECT to_regclass('qvec') IS NOT NULL AS x")["x"][0] and \
        int(q("SELECT count(*) AS n FROM qvec")["n"][0]) == len(QUERY_TEXTS):
    print("qvec already built")
else:
    run("DROP TABLE IF EXISTS qvec")
    run("CREATE TABLE qvec (id int PRIMARY KEY, text text, season int, v vector(3072))")
    t0 = time.time()
    for i, t in enumerate(QUERY_TEXTS):
        run("INSERT INTO qvec SELECT %s, %s::text, %s, ai.embedding(%s::text, %s::text)::vector",
            (i, t, seasons[i % len(seasons)], EMBED_MODEL, t))
    print(f"embedded {len(QUERY_TEXTS)} searches in {time.time() - t0:.0f}s")
if not _FIXTURE:
    run("ALTER TABLE qvec ADD COLUMN IF NOT EXISTS vh halfvec(3072)")
    run("UPDATE qvec SET vh = v::halfvec(3072) WHERE vh IS NULL")
QV = q("SELECT id, text, season FROM qvec ORDER BY id")
FILTER = "season = {s} AND recommendation = 'SIGN'"

run("SET enable_indexscan = off")
EXACT, EXACT_F, FSIZE = {}, {}, []
t0 = time.time()
for r in QV.itertuples(index=False):
    vec = f"(SELECT v FROM qvec WHERE id = {r.id})"
    EXACT[r.id] = list(q(f"SELECT report_id FROM reports ORDER BY report_embedding <=> {vec} LIMIT 10").report_id)
    f = FILTER.format(s=r.season)
    EXACT_F[r.id] = list(q(f"SELECT report_id FROM reports WHERE {f} "
                           f"ORDER BY report_embedding <=> {vec} LIMIT 10").report_id)
FSIZE = q(f"SELECT avg(n) AS a FROM (SELECT count(*) AS n FROM reports WHERE recommendation = 'SIGN' "
          f"GROUP BY season) x")["a"][0]
run("RESET enable_indexscan")
print(f"exact answers for {len(QV)} searches, plain and filtered, in {time.time() - t0:.0f}s; "
      f"a filter keeps ~{float(FSIZE):,.0f} of {N_ROWS:,} reports")
hb(S, f"{len(QV)} searches; exact answers in {time.time() - t0:.0f}s; filter keeps ~{float(FSIZE):,.0f} rows")

---
## 6 · Build both indexes

Each index gets its own copy of the table, so the planner can only ever use one index per table and
the two never compete. HNSW indexes full-precision vectors only up to 2,000 dimensions, so it is built
on the half-precision form, and its searches use the same `::halfvec(3072)` expression.

In [ ]:
S = "6 · Indexes"
LEAVES = max(1, int(math.sqrt(N_ROWS)))
IDX = {
    "ScaNN": {"table": "bench_scann", "index": "bench_scann_idx",
              "ddl": (f"CREATE INDEX bench_scann_idx ON bench_scann USING scann (report_embedding cosine) "
                      f"WITH (num_leaves = {LEAVES}, quantizer = 'sq8')"),
              "col": "report_embedding", "qcol": "v"},
    "HNSW": {"table": "bench_hnsw", "index": "bench_hnsw_idx",
             "ddl": (f"CREATE INDEX bench_hnsw_idx ON bench_hnsw USING hnsw "
                     f"((report_embedding::halfvec(3072)) halfvec_cosine_ops) "
                     f"WITH (m = {HNSW_M}, ef_construction = {HNSW_EF_CONSTRUCTION})"),
             "col": "report_embedding::halfvec(3072)", "qcol": "vh"},
}
if _FIXTURE:   # pgvector offline can't index 3072 dims; B-trees keep the flow honest
    for k, v in IDX.items():
        v["ddl"] = f"CREATE INDEX {v['index']} ON {v['table']} (report_id)"
        v["col"], v["qcol"] = "report_embedding", "v"

BUILD = {}
if RUN_INDEXES:
    run(f"SET maintenance_work_mem = '{MAINTENANCE_WORK_MEM}'")
    for name, ix in IDX.items():
        exists = bool(q("SELECT to_regclass(%s) IS NOT NULL AS x", (ix["index"],))["x"][0])
        if exists and not REBUILD:
            print(f"{name}: index exists—not rebuilt (REBUILD = True to time it again)")
            hb(S, f"{name}: exists, not rebuilt")
            continue
        run(f"DROP TABLE IF EXISTS {ix['table']}")
        run(f"CREATE TABLE {ix['table']} AS SELECT * FROM reports")
        run(f"CREATE INDEX ON {ix['table']} (season, recommendation)")
        t0 = time.time()
        try:
            run(ix["ddl"])
        except Exception as e:  # noqa: BLE001
            print(f"🔴 {name} build FAILED: {str(e)[:300]}")
            hb(S, f"{name}: build FAILED—{str(e)[:300]}")
            continue
        BUILD[name] = time.time() - t0
        run(f"ANALYZE {ix['table']}")
    run("RESET maintenance_work_mem")
for name, ix in IDX.items():
    if q("SELECT to_regclass(%s) IS NOT NULL AS x", (ix["index"],))["x"][0]:
        mb = int(q("SELECT pg_relation_size(%s) AS b", (ix["index"],))["b"][0]) / 1e6
        ix["size_mb"] = mb
        line = (f"{name}: {'built in ' + format(BUILD[name], '.0f') + 's, ' if name in BUILD else ''}"
                f"index size {mb:,.0f} MB")
        print(line)
        hb(S, line)

---
## 7 · Match recall

A faster index that finds fewer of the right answers hasn't won anything. Each index is swept over its
search setting, and the benchmark uses the **smallest setting that reaches the target recall@10** on the
96 plain searches. Filtered recall is then measured at that setting. For HNSW, filtered searches also
run with `hnsw.iterative_scan`, which keeps searching the graph when the filter discards most of what
it found.

In [ ]:
S = "7 · Recall"
SETTINGS = {"ScaNN": ("scann.num_leaves_to_search", SCANN_SWEEP), "HNSW": ("hnsw.ef_search", HNSW_SWEEP)}
CHOSEN, EXTRA = {}, {"ScaNN": [], "HNSW": []}


def search_sql(name, filtered):
    ix = IDX[name]
    where = f"WHERE {FILTER.replace('{s}', '(SELECT season FROM qvec WHERE id = %s)')} " if filtered else ""
    return (f"SELECT report_id FROM {ix['table']} {where}"
            f"ORDER BY {ix['col']} <=> (SELECT {ix['qcol']} FROM qvec WHERE id = %s) LIMIT 10")


def recall(name, filtered, sets=()):
    for k, v in sets:
        run(f"SET {k} = {v}")
    sql, rec, got, ms = search_sql(name, filtered), [], [], []
    truth = EXACT_F if filtered else EXACT
    for qid in QV.id:
        t0 = time.time()
        ids = list(q(sql, (int(qid), int(qid)) if filtered else (int(qid),)).report_id)
        ms.append((time.time() - t0) * 1000)
        rec.append(len(set(ids) & set(truth[qid])) / max(1, len(truth[qid])))
        got.append(len(ids))
    for k, _ in sets:
        run(f"RESET {k}")
    return sum(rec) / len(rec), sum(got) / len(got), sorted(ms)[len(ms) // 2]


def uses_index(name):
    plan = " ".join(q("EXPLAIN " + search_sql(name, False).replace("%s", "0")).iloc[:, 0]).lower()
    return bool(_FIXTURE) or IDX[name]["index"] in plan


if RUN_TUNING:
    for name, (knob, sweep) in SETTINGS.items():
        if "size_mb" not in IDX[name]:
            continue
        print(f"{name}: plan uses the index: {uses_index(name)}")
        hb(S, f"{name}: plan uses the index: {uses_index(name)}")
        for v in sweep:
            r, got, ms = recall(name, False, [(knob, v)])
            print(f"  {knob} = {v:<5} recall@10 {r:>5.1%}  {ms:6.1f} ms")
            hb(S, f"{name} {knob}={v}: recall@10 {r:.1%}, {ms:.1f} ms")
            if r >= TARGET_RECALL and name not in CHOSEN:
                CHOSEN[name] = v
        CHOSEN.setdefault(name, sweep[-1])
        base = [(knob, CHOSEN[name])]
        r, got, ms = recall(name, True, base)
        EXTRA[name].append(("filtered", r, got, ms))
        print(f"  chosen {knob} = {CHOSEN[name]}; filtered: recall@10 {r:.1%}, {got:.1f} rows returned, {ms:.1f} ms")
        hb(S, f"{name} chosen {knob}={CHOSEN[name]}; filtered recall@10 {r:.1%}, rows {got:.1f}, {ms:.1f} ms")
        if name == "HNSW":
            try:
                r, got, ms = recall(name, True, base + [("hnsw.iterative_scan", "relaxed_order")])
                EXTRA[name].append(("filtered, iterative scan", r, got, ms))
                print(f"  with hnsw.iterative_scan = relaxed_order: recall@10 {r:.1%}, {got:.1f} rows, {ms:.1f} ms")
                hb(S, f"HNSW filtered + iterative_scan: recall@10 {r:.1%}, rows {got:.1f}, {ms:.1f} ms")
            except Exception as e:  # noqa: BLE001
                print(f"  hnsw.iterative_scan unavailable: {str(e)[:150]}")
                hb(S, f"HNSW iterative_scan unavailable: {str(e)[:150]}")

---
## 8 · Under load

For each index, plain and filtered: `N` workers, each on its own connection with the chosen search
setting, run searches back to back for `STEP_SECONDS`, each picking one of the 96 queries at random.
Reported per step: queries per second, latency percentiles, and errors.

The workers run in this notebook. If the notebook itself becomes the bottleneck, the **client CPU**
column approaches 100% of one core and the numbers above that point say more about Python than about
AlloyDB; the handback flags it.

In [ ]:
import random
S = "8 · Load"
LOAD = []


def _worker(name, filtered, setting, stop_at, warm_until, out, errs):
    try:
        conn = connect()
        cur = conn.cursor()
        knob = SETTINGS[name][0]
        cur.execute(f"SET {knob} = {setting}")
        if filtered and name == "HNSW" and HNSW_ITER:
            cur.execute("SET hnsw.iterative_scan = relaxed_order")
        sql = search_sql(name, filtered)
        rnd = random.Random()
        while time.time() < stop_at:
            qid = int(rnd.choice(QIDS))
            t0 = time.time()
            cur.execute(sql, (qid, qid) if filtered else (qid,))
            cur.fetchall()
            if t0 >= warm_until:
                out.append((time.time() - t0) * 1000)
        cur.close(); conn.close()
    except Exception as e:  # noqa: BLE001
        errs.append(f"{type(e).__name__}: {str(e)[:150]}")


def step(name, filtered, n):
    out, errs = [], []
    warm_until = time.time() + 5 + n * 0.05            # connections open during the warm-up
    stop_at = warm_until + STEP_SECONDS
    cpu0, t0 = time.process_time(), time.time()
    ths = [threading.Thread(target=_worker, args=(name, filtered, CHOSEN[name], stop_at, warm_until, out, errs))
           for _ in range(n)]
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    cpu = (time.process_time() - cpu0) / max(time.time() - t0, 1)
    lat = sorted(out)
    pct = (lambda p: lat[min(len(lat) - 1, int(p * len(lat)))] if lat else float("nan"))
    row = {"index": name, "filtered": filtered, "clients": n, "qps": len(lat) / STEP_SECONDS,
           "p50": pct(0.50), "p95": pct(0.95), "p99": pct(0.99), "errors": len(errs), "client_cpu": cpu}
    LOAD.append(row)
    flag = "  ⚠️ client-bound" if cpu > 0.85 else ""
    line = (f"{name:<5} {'filtered' if filtered else 'plain   '} {n:>4} clients: {row['qps']:>7.0f} q/s  "
            f"p50 {row['p50']:6.1f}  p95 {row['p95']:6.1f}  p99 {row['p99']:6.1f} ms  "
            f"errors {len(errs)}  client CPU {cpu:.0%}{flag}")
    print(line)
    hb(S, line + (f"—{errs[0]}" if errs else ""))


QIDS = list(QV.id)
HNSW_ITER = any(lbl == "filtered, iterative scan" for lbl, *_ in EXTRA["HNSW"])
if not RUN_LOAD:
    print("skipped—RUN_LOAD is False")
elif len(CHOSEN) < 2:
    print("skipped—needs both indexes tuned in section 7")
    hb(S, "skipped")
else:
    print(f"settings: {CHOSEN}; HNSW filtered uses iterative scan: {HNSW_ITER}\n")
    hb(S, f"settings {CHOSEN}; HNSW filtered iterative scan: {HNSW_ITER}; {STEP_SECONDS}s per step")
    for filtered in (False, True):
        for name in ("ScaNN", "HNSW"):
            for n in CLIENT_STEPS:
                step(name, filtered, n)
        print()

---
## 9 · Summary and handback

The comparison side by side, then one box with everything Claude needs and a **Copy** button. The
results are also saved as JSON under `benchmarks/` in the course bucket.

In [ ]:
import html as _html
from IPython.display import HTML, display

if LOAD:
    L = pd.DataFrame(LOAD)
    for filtered in (False, True):
        sub = L[L.filtered == filtered].pivot(index="clients", columns="index", values=["qps", "p95"])
        print(("FILTERED" if filtered else "PLAIN") + " searches—queries/s and p95 ms")
        print(sub.round(1).to_string(), "\n")
    peak = L.groupby(["index", "filtered"]).qps.max().round(0).to_dict()
    hb("Summary", "peak q/s: " + "; ".join(f"{k[0]} {'filtered' if k[1] else 'plain'} {v:,.0f}" for k, v in peak.items()))
hb("Summary", "index size MB: " + ", ".join(f"{k} {v.get('size_mb', 0):,.0f}" for k, v in IDX.items()))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = {"run_utc": stamp, "project": PROJECT, "shape": SHAPE, "runtime_vcpu": os.cpu_count(), "versions": VER,
       "rows": N_ROWS, "build_s": BUILD, "sizes_mb": {k: v.get("size_mb") for k, v in IDX.items()},
       "chosen": CHOSEN, "filtered_recall": EXTRA, "load": LOAD, "target_recall": TARGET_RECALL}
p = BASE / f"index-benchmark-{stamp}.json"
p.write_text(json.dumps(OUT, indent=2, default=str))
r = gcs("cp", str(p), f"{RESULTS_PREFIX}/{p.name}")
saved = r.returncode == 0
print(f"results {'saved to ' + RESULTS_PREFIX + '/' + p.name if saved else 'NOT saved to the bucket: ' + r.stderr[:200]}")

lines = ["=" * 70, "CYMBALGOAL INDEX BENCHMARK—03 HANDBACK", f"run {stamp}", "=" * 70]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
writes = [c for c in GCS_LOG if c.split()[2] == "cp"]
bad = [c for c in writes if not any(_under(c.split()[-1], w) for w in WRITE_OK)]
lines += ["", "## Bucket audit",
          f"calls {len(GCS_LOG)}; writes {len(writes)}, all allowed: {not bad}; results saved: {saved}", "=" * 70]
TEXT = "\n".join(lines)
(BASE / "handback-03.txt").write_text(TEXT)
assert not bad, f"a write left the allowed prefixes: {bad}"

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:100%">
  <button id="cg-copy" style="font-size:15px;font-weight:700;padding:10px 20px;border-radius:8px;border:0;
      background:#1f8a6d;color:#fff;cursor:pointer">Copy handback</button>
  <span id="cg-msg" style="margin-left:12px;font-weight:600;color:#1f8a6d"></span>
  <textarea id="cg-box" readonly style="display:block;width:100%;height:340px;margin-top:10px;
      font:12px/1.4 ui-monospace,Menlo,monospace;padding:10px;box-sizing:border-box">{_html.escape(TEXT)}</textarea>
</div>
<script>
(function() {{
  const b = document.getElementById("cg-copy"), t = document.getElementById("cg-box"), m = document.getElementById("cg-msg");
  b.addEventListener("click", async () => {{
    try {{ await navigator.clipboard.writeText(t.value); m.textContent = "Copied—paste it into the chat."; }}
    catch (e) {{ t.focus(); t.select(); m.textContent = "Clipboard blocked: the text is selected, use Edit → Copy."; }}
  }});
}})();
</script>'''))
print(f"{len(TEXT):,} characters; also saved to {BASE / 'handback-03.txt'}")